In [ ]:
#| default_exp eddb.edsm

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

#pylint: disable=missing-module-docstring
#pylint: disable=missing-function-docstring
#pylint: disable=invalid-name

from functools import lru_cache
import requests, json
import os, time, logging
import numpy as np
import edcompanion.core



In [ ]:
edcompanion.core.init_console_logging(__name__)

2026-02-05T14:17:36+0100 INFO	43372	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| export

syslog = logging.getLogger(__name__)
configuration = edcompanion.core.configuration['EDSM']

In [ ]:
#| export

def get_commander_position(commander_name=configuration['commander_name'], token=configuration['token'], verbose=True):
    req = requests.get(
        f"{configuration['api_logs_v1']}/get-position",
        params=dict(
            commanderName=commander_name,
            apiKey=token,
            showCoordinates=1)
        )
    if req.status_code == 200:
        record = req.json()
        if verbose:
            return record if record else {}
        return record if record else {}
    else:
        syslog.error(req.status_code)
        return {'text': str(req.text)}


In [ ]:
record = get_commander_position(verbose=True)

In [ ]:
record

{'msgnum': 100,
 'msg': 'OK',
 'system': 'Shinrarta Dezhra',
 'firstDiscover': False,
 'date': '2024-10-15 19:11:43',
 'coordinates': {'x': 55.71875, 'y': 17.59375, 'z': 27.15625},
 'isDocked': False,
 'dateLastActivity': '2026-02-05 13:03:48',
 'url': 'https://www.edsm.net/en/user/profile/id/159706/cmdr/immerlicht'}

In [ ]:

#| exporti
edsm_discard = set(["JournalFinished"])


In [ ]:

#| export
def get_api_discarded():
    req = requests.get(f'{configuration["api_journal_v1"]}/discard')
    if req.status_code == 200:
        return edsm_discard | set(req.json())

    return edsm_discard


In [ ]:

#| export
def post_journal_item(journal_item, commander_name=os.getenv("EDSM_USER"), token=os.getenv(key="EDSM_TOKEN"), software_info={
                    "fromSoftware":"EDCompanion",
                    "fromSoftwareVersion":"1.0",
                    "fromGameVersion":"4.0.0.1809",
                    "fromGameBuild":"r305601/r0 ",
                }):
    if journal_item.get('event') in edsm_discard or "fromGameVersion" not in software_info:
        return {}

    time.sleep(0.1)
    post_body = dict(
                commanderName=commander_name,
                apiKey=token,
                **software_info,
                message=journal_item
            )

    req = requests.post(
        'https://www.edsm.net/api-journal-v1/',
        json=post_body
    )
    if req.status_code == 200:
        #text = req.text
        record = req.json()
        return record if record else {}
    else:
        return {}


In [ ]:

#| export
@lru_cache(512)
def get_edsm_info(systemname, verbose=True):
    '''Retrieves information about a named system. '''
    if not systemname:
        return {}
    req = requests.get(
        'https://www.edsm.net/api-system-v1/bodies' if verbose else 'https://www.edsm.net/api-v1/system',
        params=dict(
            systemName=systemname,
            showCoordinates=1)
        )
    if req.status_code == 200:
        record = req.json()
        syslog.info(req.headers)
        return record if record else {}
    else:
        return {}


In [ ]:
system_info = get_edsm_info(record.get('system'), True)

2025-12-28T11:11:13+0100 INFO	35612	__main__	3903079254.py	get_edsm_info	15	{'Server': 'nginx', 'Date': 'Sun, 28 Dec 2025 10:11:13 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Methods': 'GET, POST', 'Access-Control-Allow-Headers': 'Content-Type', 'X-Powered-By': 'EDSM.NET APIv1', 'X-Rate-Limit-Limit': '720', 'X-Rate-Limit-Remaining': '719', 'X-Rate-Limit-Reset': '5', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload'}


In [ ]:
list(system_info.keys())

['id', 'id64', 'name', 'url', 'bodyCount', 'bodies']

In [ ]:
system_info

{'id': 4345,
 'id64': 3932277478106,
 'name': 'Shinrarta Dezhra',
 'url': 'https://www.edsm.net/en/system/bodies/id/4345/name/Shinrarta+Dezhra',
 'bodyCount': 39,
 'bodies': [{'id': 14923,
   'id64': 36032729296442074,
   'bodyId': 1,
   'name': 'Shinrarta Dezhra',
   'discovery': {'commander': 'Redfoxxx', 'date': '2017-09-26 20:14:16'},
   'type': 'Star',
   'subType': 'K (Yellow-Orange) Star',
   'parents': [{'Null': 0}],
   'distanceToArrival': 0,
   'isMainStar': True,
   'isScoopable': True,
   'age': 8068,
   'spectralClass': 'K5',
   'luminosity': 'V',
   'absoluteMagnitude': 7.129517,
   'solarMasses': 0.648438,
   'solarRadius': 0.7666015240833932,
   'surfaceTemperature': 4343,
   'orbitalPeriod': 1847.1559440648148,
   'semiMajorAxis': 1.156960272745914,
   'orbitalEccentricity': 0.018651,
   'orbitalInclination': 21.118831,
   'argOfPeriapsis': 201.271868,
   'rotationalPeriod': 3.305768617083334,
   'rotationalPeriodTidallyLocked': False,
   'axialTilt': -0.041915,
   'bel

In [ ]:

#| export
@lru_cache(64)
def get_edsm_system_risk(systemname):
    if not systemname:
        return {}
    traffic = requests.get(
        'https://www.edsm.net/api-system-v1/traffic',
        params=dict(systemName=systemname)
    )

    if traffic.status_code == 200:
        trafficrecord = traffic.json().get('traffic',{})
        weektraffic = trafficrecord.get('week', 0)
        if weektraffic > 0:
            deaths = requests.get(
                'https://www.edsm.net/api-system-v1/deaths',
                params=dict(systemName=systemname)
            )

            if deaths.status_code == 200:
                deathsrecord = deaths.json().get('deaths',{})

                totaldeaths = deathsrecord.get('total',0)
                if totaldeaths > 0:
                    weekdeaths = deathsrecord.get('week',0)
                    totaltraffic = trafficrecord.get('total', 0)
                    return (weekdeaths*totaltraffic) / (weektraffic * totaldeaths) if weekdeaths > 0 else totaldeaths/totaltraffic

        return 0

    return 0


In [ ]:
get_edsm_system_risk('Shinrarta Dezhra')

0.1391607571996052

In [ ]:

#| export
@lru_cache(512)
def distance_between_systems(s1name,s2name):
    s1 = get_edsm_info(s1name, verbose=False)
    s2 = get_edsm_info(s2name, verbose=False)
    c1 = np.asarray([s1.get('coords',dict(x=0,y=0,z=0)).get(k) for k in ['x', 'y', 'z']])
    c2 = np.asarray([s2.get('coords',dict(x=0,y=0,z=0)).get(k) for k in ['x', 'y', 'z']])
    return np.sqrt(np.sum(np.square(c1-c2)))

def get_systems_in_cube(system, size=100):
    base_url='https://www.edsm.net/api-v1/cube-systems'

    if isinstance(system, str):
        return get_systems_in_cube_by_name(system, size)

    req = requests.get(
        base_url,
        params=dict(
            x=system[0],
            y=system[1],
            z=system[2],
            showCoordinates=1,
            showPrimaryStar=1,
            size=size
        )
    )
    if req.status_code == 200:
        return req.json()
    else:
        return {}


In [ ]:

#| export
@lru_cache(32)
def get_systems_in_cube_by_name(system, size=100):
    base_url='https://www.edsm.net/api-v1/cube-systems'
    if not system:
        return {}
    req = requests.get(
        base_url,
        params=dict(
            systemName=system,
            showCoordinates=1,
            showPrimaryStar=1,
            size=size
        )
    )
    syslog.info(req.headers)
    if req.status_code == 200:
        return req.json()
    else:
        return {}


In [ ]:
get_systems_in_cube_by_name('Shinrarta Dezhra', 50)

[{'distance': 31.85,
  'bodyCount': 23,
  'name': 'NN 3709',
  'coords': {'x': 30.75, 'y': 29.4375, 'z': 11.3125},
  'coordsLocked': True,
  'primaryStar': {'type': 'M (Red dwarf) Star',
   'name': 'NN 3709 A',
   'isScoopable': True}},
 {'distance': 29.01,
  'bodyCount': 17,
  'name': 'LTT 5419',
  'coords': {'x': 30.96875, 'y': 29.28125, 'z': 36.78125},
  'coordsLocked': True,
  'primaryStar': {'type': 'M (Red dwarf) Star',
   'name': 'LTT 5419 A',
   'isScoopable': True}},
 {'distance': 27.94,
  'bodyCount': 4,
  'name': 'Crucis Sector AQ-Y c24',
  'coords': {'x': 31.59375, 'y': 29.3125, 'z': 35},
  'coordsLocked': True,
  'primaryStar': {'type': 'K (Yellow-Orange) Star',
   'name': 'Crucis Sector AQ-Y c24 A',
   'isScoopable': True}},
 {'distance': 24.71,
  'bodyCount': 17,
  'name': 'Anyanwu',
  'coords': {'x': 32.15625, 'y': 23.09375, 'z': 32.1875},
  'coordsLocked': True,
  'primaryStar': {'type': 'K (Yellow-Orange) Star',
   'name': 'Anyanwu',
   'isScoopable': True}},
 {'dista

In [ ]:

#| export
@lru_cache(32)
def get_systems_in_sphere(system, radius=100):
    base_url='https://www.edsm.net/api-v1/sphere-systems'
    if not system:
        return {}
    req = requests.get(
        base_url,
        params=dict(
            systemName=system,
            showCoordinates=1,
            showPrimaryStar=1,
            radius=radius
        )
    )
    syslog.info(req.headers)
    if req.status_code == 200:
        return req.json()
    else:
        return {}


In [ ]:

#| exporti
edsm_discard |= set([
    "ShutDown", "EDDItemSet", "EDDCommodityPrices", "ModuleArrived", "ShipArrived", "Coriolis", "EDShipyard", "Market", "Shipyard",    "Outfitting", "ModuleInfo", "Status", "SquadronCreated", "SquadronStartup", "DisbandedSquadron", "InvitedToSquadron", "AppliedToSquadron",    "JoinedSquadron", "LeftSquadron",    "SharedBookmarkToSquadron", "CarrierStats",    "CarrierTradeOrder", "CarrierFinance",    "CarrierBankTransfer", "CarrierCrewServices",    "CarrierJumpRequest", "CarrierJumpCancelled",    "CarrierDepositFuel", "CarrierDockingPermission",    "CarrierModulePack", "CarrierBuy", "CarrierNameChange", "CarrierDecommission", "BookDropship", "CancelDropship", "DropshipDeploy", "CollectItems", "DropItems", "Disembark", "Embark", "Fileheader", "Commander", "NewCommander", "ClearSavedGame", "Music", "Continued", "Passengers", "DockingCancelled", "DockingDenied", "DockingGranted", "DockingRequested", "DockingTimeout", "StartJump", "Touchdown", "Liftoff", "NavBeaconScan", "SupercruiseEntry", "SupercruiseExit", "NavRoute", "NavRouteClear", "PVPKill", "CrimeVictim", "UnderAttack", "ShipTargeted", "Scanned", "DataScanned", "DatalinkScan", "EngineerApply", "EngineerLegacyConvert", "FactionKillBond", "Bounty", "CapShipBond", "DatalinkVoucher", "SystemsShutdown", "EscapeInterdiction", "HeatDamage", "HeatWarning", "HullDamage", "ShieldState", "FuelScoop", "LaunchDrone", "AfmuRepairs", "CockpitBreached", "ReservoirReplenished", "CargoTransfer", "ApproachBody", "LeaveBody", "DiscoveryScan", "MaterialDiscovered", "Screenshot", "CrewAssign", "CrewFire", "NpcCrewRank", "ShipyardNew", "StoredModules", "MassModuleStore", "ModuleStore", "ModuleSwap", "SuitLoadout", "SwitchSuitLoadout", "CreateSuitLoadout", "LoadoutEquipModule", "PowerplayVote", "PowerplayVoucher", "ChangeCrewRole", "CrewLaunchFighter", "CrewMemberJoins", "CrewMemberQuits", "CrewMemberRoleChange", "KickCrewMember", "EndCrewSession", "LaunchFighter", "DockFighter", "FighterDestroyed", "FighterRebuilt", "VehicleSwitch", "LaunchSRV", "DockSRV", "SRVDestroyed", "JetConeBoost", "JetConeDamage", "RebootRepair", "RepairDrone", "WingAdd", "WingInvite", "WingJoin", "WingLeave", "ReceiveText", "SendText", "Shutdown", "FSSSignalDiscovered", "AsteroidCracked", "ProspectedAsteroid", "ScanBaryCentre", "FSSBodySignals", "SAASignalsFound", "ScanOrganic", "JournalFinished"
])

